## Traditional Attempt at classification (Random Forest & XGBoost)

This is a first attempt at classification of the combined dataset using Random Forest.

Reading in the data

In [13]:
#Bad things happen when you uncomment these lines
# Beware
#df = pd.read_parquet('../data/MERGED_LCS.parquet')
#df

Reducing the precision of the timestamps and flux values so my kernel stops crashing

In [1]:
import pyarrow.parquet as pq
import pyarrow as pa
import numpy as np
import pandas as pd

In [2]:

infile = "../data/MERGED_LCS.parquet"
outfile = "../data/MERGED_LCS_reduced.parquet"

def round_array(x, ndigits=5):
    if x is None:
        return None
    arr = np.asarray(x, dtype=np.float32)
    return np.round(arr, ndigits).tolist()

pf = pq.ParquetFile(infile)
writer = None

for batch in pf.iter_batches(batch_size=16, columns=["time", "flux", "Class", "KIC"]):
    # Convert directly from Arrow to a normal Python dict
    data = batch.to_pydict()

    # Build the dataframe explicitly, including KIC
    df = pd.DataFrame({
        "time": data["time"],
        "flux": data["flux"],
        "Class": data["Class"],
        "KIC": data["KIC"],
    })

    df["time"] = df["time"].apply(round_array)
    df["flux"] = df["flux"].apply(round_array)

    # Save this batch
    table = pa.Table.from_pandas(df, preserve_index=False)

    if writer is None:
        writer = pq.ParquetWriter(outfile, table.schema, compression="zstd")

    writer.write_table(table)

if writer is not None:
    writer.close()

In [2]:
df = pd.read_parquet("../data/MERGED_LCS_reduced.parquet")

In [3]:
df

,time,flux,Class,KIC
0,"[131.51271057128906, 131.53314208984375, 131.5...","[1.0392199754714966, 1.0383399724960327, 1.038...",CONFIRMED,757450
1,"[352.39654541015625, 352.43743896484375, 352.4...","[1.019569993019104, 1.018839955329895, 1.01899...",FALSE POSITIVE,892772
2,"[120.539306640625, 120.55975341796875, 120.580...","[1.0488500595092773, 1.0523099899291992, 1.051...",CANDIDATE,1025986
3,"[131.51271057128906, 131.53314208984375, 131.5...","[1.0509400367736816, 1.0503300428390503, 1.050...",FALSE POSITIVE,1026032
4,"[120.53929901123047, 120.55973815917969, 120.5...","[1.059939980506897, 1.0591700077056885, 1.0589...",CONFIRMED,1026957
...,...,...,...,...
18877,"[1940.0084228515625, 1940.02880859375, 1940.04...","[0.9961699843406677, 0.9965299963951111, 0.996...",VARIABLE STAR,202140012
18878,"[1940.0084228515625, 1940.02880859375, 1940.04...","[0.9897400140762329, 0.9907699823379517, 0.991...",VARIABLE STAR,202140013
18879,"[1940.0089111328125, 1940.029296875, 1940.0498...","[1.0229099988937378, 1.014359951019287, 1.0042...",FALSE POSITIVE,202140059
18880,"[1940.0089111328125, 1940.029296875, 1940.0498...","[1.0080900192260742, 1.0085899829864502, 1.009...",FALSE POSITIVE,202140094


In [4]:
df.duplicated(subset="KIC").sum()

np.int64(1722)

In [5]:
len(df['KIC'].unique())

17160

## Preprocessing

In [4]:
def chooseLabel(labels):
    labels = list(labels)
    nonFp = [lab for lab in labels if lab != "FALSE POSITIVE"]

    if len(nonFp) > 0:
        # If there are multiple non-FP labels, take the most common one
        return pd.Series(nonFp).mode().iloc[0]

    # If all labels are FALSE POSITIVE, keep one of them
    return pd.Series(labels).mode().iloc[0]

In [5]:
df = (
    df.groupby("KIC", as_index=False)
        .agg({
            "time": "first",
            "flux": "first",
            "Class": chooseLabel
        })
)

In [6]:
df['Class'].value_counts()

Class
FALSE POSITIVE           7522
ECLIPSING BINARY STAR    3049
VARIABLE STAR            2980
CONFIRMED                1972
CANDIDATE                1637
Name: count, dtype: int64

In [7]:
df = df[df['Class'] != 'CANDIDATE'].copy()

In [10]:
df.duplicated(subset="KIC").sum()

np.int64(0)

In [8]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy.ndimage import median_filter

def sigma_clip_mad(t, f, sigma=5.0):
    """
    Robust outlier removal using MAD.
    Returns filtered time and flux arrays.
    """
    if len(f) == 0:
        return t, f

    med = np.median(f)
    mad = np.median(np.abs(f - med))

    if mad == 0:
        return t, f

    robust_sigma = 1.4826 * mad
    keep = np.abs(f - med) <= sigma * robust_sigma
    return t[keep], f[keep]

def normalize_by_median(f):
    """
    Normalize flux by its median.
    """
    med = np.median(f)
    if med == 0:
        return f
    return f / med

def detrend_with_median_filter(t, f, window_frac=0.1):
    """
    Detrend by interpolating onto a uniform grid and subtracting a median-filter trend.
    Works best after normalization.
    """
    n = len(f)
    if n < 10:
        return f

    # Interpolate onto a uniform grid
    grid = np.linspace(t.min(), t.max(), n, dtype=np.float32)
    f_interp = np.interp(grid, t, f).astype(np.float32)

    # Median filter window size
    win = max(5, int(n * window_frac))
    if win >= n:
        win = n - 1
    if win % 2 == 0:
        win -= 1
    if win < 5:
        return f

    trend = median_filter(f_interp, size=win, mode="nearest")
    trend_at_t = np.interp(t, grid, trend).astype(np.float32)

    # Avoid divide-by-zero
    trend_at_t = np.where(np.abs(trend_at_t) < 1e-8, 1.0, trend_at_t)

    # Since normalized flux is usually around 1, division is okay here
    return f / trend_at_t


def preprocess_lightcurve(time, flux, sigma=5.0, detrend=True, n_points=200):
    t = np.asarray(time, dtype=np.float32)
    f = np.asarray(flux, dtype=np.float32)

    mask = np.isfinite(t) & np.isfinite(f)
    t = t[mask]
    f = f[mask]

    if len(t) < 2:
        return np.zeros(n_points, dtype=np.float32)

    order = np.argsort(t)
    t = t[order]
    f = f[order]

    f = normalize_by_median(f)
    t, f = sigma_clip_mad(t, f, sigma=sigma)

    if len(t) < 2:
        return np.zeros(n_points, dtype=np.float32)

    if detrend and len(t) >= 10:
        f = detrend_with_median_filter(t, f)

    # If detrending or clipping produced bad values
    mask = np.isfinite(t) & np.isfinite(f)
    t = t[mask]
    f = f[mask]

    if len(t) < 2:
        return np.zeros(n_points, dtype=np.float32)

    t = (t - t.min()) / (t.max() - t.min() + 1e-8)
    t_new = np.linspace(0, 1, n_points, dtype=np.float32)
    f_new = np.interp(t_new, t, f).astype(np.float32)

    return f_new

def extra_features(vec):
    vec = np.asarray(vec, dtype=np.float32)
    d = np.diff(vec)

    feats = [
        np.mean(vec),
        np.std(vec),
        np.min(vec),
        np.max(vec),
        np.median(vec),
        np.percentile(vec, 25),
        np.percentile(vec, 75),
        np.ptp(vec),
        np.mean(d) if len(d) else 0.0,
        np.std(d) if len(d) else 0.0,
        np.mean(np.abs(d)) if len(d) else 0.0,
    ]

    return np.array(feats, dtype=np.float32)

def fft_features(vec):
    vec = np.asarray(vec, dtype=np.float32)
    fft = np.fft.rfft(vec)
    mag = np.abs(fft)

    feats = [
        np.max(mag) if len(mag) else 0.0,
        np.mean(mag) if len(mag) else 0.0,
        np.std(mag) if len(mag) else 0.0,
        np.sum(mag) if len(mag) else 0.0,
    ]
    return np.array(feats, dtype=np.float32)

def build_feature_vector(time, flux, n_points=200):
    vec = preprocess_lightcurve(time, flux, n_points=n_points)

    if len(vec) != n_points:
        vec = np.zeros(n_points, dtype=np.float32)

    feats = np.concatenate([
        vec,
        extra_features(vec),
        fft_features(vec),
    ]).astype(np.float32)

    return feats

In [9]:
cleaned_df = df.copy()

cleaned_df["features"] = cleaned_df.apply(
    lambda row: preprocess_lightcurve(row["time"], row["flux"]),
    axis=1
)

# keep only what you need
cleaned_df = cleaned_df[["features", "Class", "KIC"]]

# Drop any bad rows just in case
cleaned_df = cleaned_df[cleaned_df["features"].apply(lambda x: isinstance(x, np.ndarray) and len(x) > 0)].reset_index(drop=True)

X = np.vstack(cleaned_df["features"].values)
y = cleaned_df["Class"].values
kic = cleaned_df["KIC"].values

print("X shape:", X.shape)
print("Classes:", pd.Series(y).value_counts())

X shape: (15523, 200)
Classes: FALSE POSITIVE           7522
ECLIPSING BINARY STAR    3049
VARIABLE STAR            2980
CONFIRMED                1972
Name: count, dtype: int64


In [10]:
cleaned_df

,features,Class,KIC
0,"[1.0, 1.0142267, 0.99171585, 1.0041214, 1.0052...",CONFIRMED,757450
1,"[1.0, 0.9997978, 0.9993214, 1.0000552, 0.99973...",FALSE POSITIVE,892772
2,"[1.0, 0.99873066, 1.0031911, 1.0013676, 1.0008...",ECLIPSING BINARY STAR,1026032
3,"[1.0, 1.0000204, 0.99358475, 1.0007892, 0.9877...",CONFIRMED,1026957
4,"[1.0, 1.0006799, 1.0002626, 1.0004326, 0.99993...",FALSE POSITIVE,1027438
...,...,...,...
15518,"[1.0, 1.0007917, 1.0007094, 1.0013616, 1.00050...",VARIABLE STAR,202140012
15519,"[1.0, 0.9977387, 1.0011798, 1.000965, 0.991027...",VARIABLE STAR,202140013
15520,"[1.0, 1.0157899, 1.0060803, 1.0206845, 1.02289...",FALSE POSITIVE,202140059
15521,"[1.0, 0.99416083, 0.9960654, 0.9960163, 0.9942...",FALSE POSITIVE,202140094


In [11]:
try:
    from imblearn.over_sampling import RandomOverSampler
    HAS_IMBLEARN = True
except ImportError:
    HAS_IMBLEARN = False

# Random Forest Model

In [12]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Build X and y
X = np.vstack(cleaned_df["features"].values)
y = cleaned_df["Class"].values
kic = cleaned_df["KIC"].values   # keep for reference, not training

# Encode labels
le = LabelEncoder()
y_enc = le.fit_transform(y)

# Train/test split
X_train, X_test, y_train, y_test, kic_train, kic_test = train_test_split(
    X, y_enc, kic,
    test_size=0.2,
    random_state=42,
    stratify=y_enc
)
# Balance training data if possible
if HAS_IMBLEARN:
    ros = RandomOverSampler(random_state=42)
    X_train, y_train = ros.fit_resample(X_train, y_train)

# Tuned Random Forest
rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=20,
    min_samples_leaf=2,
    min_samples_split=4,
    max_features="sqrt",
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=42
)

rf.fit(X_train, y_train)

# Evaluate
y_pred = rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

Python(5950) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Accuracy: 0.6135265700483091

Classification report:
                       precision    recall  f1-score   support

            CONFIRMED       0.41      0.23      0.30       394
ECLIPSING BINARY STAR       0.66      0.62      0.64       610
       FALSE POSITIVE       0.61      0.80      0.69      1505
        VARIABLE STAR       0.70      0.39      0.50       596

             accuracy                           0.61      3105
            macro avg       0.60      0.51      0.53      3105
         weighted avg       0.61      0.61      0.59      3105


Confusion matrix:
[[  92   49  253    0]
 [  17  380  207    6]
 [ 117   96 1200   92]
 [   0   47  316  233]]


In [13]:
from sklearn.metrics import f1_score

print("Macro F1:", f1_score(y_test, y_pred, average="macro"))

Macro F1: 0.5329765302831231
